# DenseNet-161 on CIFAR-100: scratch vs ImageNet transfer

This notebook explains the matched transfer-learning experiment implemented in `mlshowcase/transfer.py` and reads the evidence produced by the command-line run:

```sh
uv run --locked --extra deep-learning python -m mlshowcase.transfer --epochs 1 --seeds 42 --data-dir data --device cpu
```

The device default is `auto` (CUDA when available, otherwise CPU) and the measured evidence was produced on CPU, pinned with `--device cpu`.

- a from-scratch DenseNet-161 (`weights=None`) and an ImageNet-1K DenseNet-161 with a frozen backbone and a fresh 100-class head;
- the same stratified carve of the official CIFAR-100 training split, 45000 train / 5000 dev, with the official 10000-image test split held out until selection;
- the same 64 px preprocessing derived from the DenseNet-161 weights recipe, plus matched SGD settings (lr 0.01, momentum 0.9, batch 32) under a fixed one-epoch default budget;
- checkpoint selection by development macro F1, then one evaluation on the official test split.

*Run All* reads `results/transfer.json` and the two generated PNG figures. It never retrains a model, downloads data, or downloads weights.

In [ ]:
import json
import os
import sys
import tomllib
from pathlib import Path

# This notebook runs with its checkout's ``notebooks/`` directory as the working
# directory, so ``..`` is the public checkout root. Set MLSHOWCASE_ROOT to the
# checkout's absolute path to start the notebook from anywhere else.
EXPECTED_PROJECT = "mlshowcase"


def require_showcase_checkout(root: Path, *required_files: str) -> None:
    """Fail before any import or result read unless ``root`` is this checkout."""
    guide = (
        f"{root} is not the mlshowcase checkout this notebook belongs to: run the "
        "notebook with its notebooks/ directory as the working directory, or set "
        "MLSHOWCASE_ROOT to the checkout's absolute path"
    )
    pyproject = root / "pyproject.toml"
    if not pyproject.is_file():
        raise FileNotFoundError(f"{guide} (no pyproject.toml found there)")
    try:
        name = tomllib.loads(pyproject.read_text(encoding="utf-8"))["project"]["name"]
    except (KeyError, tomllib.TOMLDecodeError) as error:
        raise ValueError(f"{guide} (project.name could not be read)") from error
    if name != EXPECTED_PROJECT:
        raise ValueError(f"{guide} (project.name is {name!r})")
    for relative in required_files:
        if not (root / relative).is_file():
            raise FileNotFoundError(f"{guide} (missing {relative})")


_override = os.environ.get("MLSHOWCASE_ROOT")
if _override and not Path(_override).is_absolute():
    raise ValueError(f"MLSHOWCASE_ROOT must be an absolute path, not {_override!r}")
SHOWCASE_ROOT = Path(_override).resolve() if _override else Path("..").resolve()
require_showcase_checkout(SHOWCASE_ROOT, "mlshowcase/transfer.py", "notebooks/transfer.ipynb")

if str(SHOWCASE_ROOT) not in sys.path:
    sys.path.insert(0, str(SHOWCASE_ROOT))

RESULTS_JSON = SHOWCASE_ROOT / "results" / "transfer.json"

In [ ]:
if not RESULTS_JSON.is_file():
    raise FileNotFoundError(
        f"missing {RESULTS_JSON.relative_to(SHOWCASE_ROOT)}; generate the evidence first with "
        "`uv run --locked --extra deep-learning python -m mlshowcase.transfer` from the repository root"
    )

payload = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))
runs = payload["runs"]
print(
    f"loaded {RESULTS_JSON.relative_to(SHOWCASE_ROOT)}: "
    + ", ".join(f"{len(records)} {variant} run(s)" for variant, records in runs.items())
)

## What the evidence file records

Every number below is read from the generated JSON; nothing is recomputed here.

| Field | Contents |
| --- | --- |
| `protocol.provenance` | CIFAR-100 loader, official split sizes, initial weights, coursework note |
| `protocol.splits` | stratified 90/10 strategy, split seed, per-split and per-class counts |
| `protocol.preprocessing` | the exact shared weights transform: crop, resize, interpolation, mean, std |
| `protocol.seeds` | model/shuffle seeds and which generator each one seeds |
| `protocol.hyperparameters` | SGD learning rate, momentum, batch size, epochs, loss, thread count |
| `protocol.selection` | development macro-F1 criterion, tie-break, held-out discipline |
| `protocol.budget` | the fixed-budget disclosure (one epoch by default) |
| `protocol.freezing` | what is frozen for the pretrained variant and the unequal-parameter caveat |
| `runs` | per-variant, per-seed histories, selections, final test metrics, timings, parameter counts |
| `summary` | repeat statistics across seeds and the paired pretrained-minus-scratch delta |
| `environment` | public-safe Python/OS/CPU/package metadata (no paths or host names) |

In [ ]:
protocol = payload["protocol"]
splits = protocol["splits"]
preprocessing = protocol["preprocessing"]

print("strategy   :", splits["strategy"])
print("counts     :", splits["counts"])
print("policy     :", splits["policy"])
print(
    "preprocess :",
    preprocessing["crop_size"],
    "px crop /",
    preprocessing["resize_size"],
    "px resize,",
    preprocessing["interpolation"],
    "interpolation",
)
print("normalize  : mean", preprocessing["mean"], "std", preprocessing["std"])
print("weights    :", protocol["provenance"]["initial_weights"]["pretrained"]["name"])
print(
    "seeds      :",
    protocol["seeds"]["model_and_shuffle"],
    "(split seed",
    str(protocol["seeds"]["split"]) + ")",
)
print("settings   :", protocol["hyperparameters"])
print("selection  :", protocol["selection"]["criterion"], "->", protocol["selection"]["held_out"])
print("budget     :", protocol["budget"]["policy"])

In [ ]:
print(json.dumps(payload["environment"], indent=2, sort_keys=True))

In [ ]:
for part in ("train", "dev", "test"):
    class_counts = splits["class_counts"][part]
    values = [class_counts[str(index)] for index in range(len(class_counts))]
    print(
        f"{part:5s}: {len(values)} classes, "
        f"{min(values)}-{max(values)} images per class, {sum(values)} images total"
    )

## What this rebuild changes from the original coursework

The historical Assignment 9 notebook is the source of the research question, not of these results. This rebuild keeps the question and corrects the methodological defects of the historical implementation instead of copying them; `tests/test_transfer.py` pins the corrected behaviour.

- **Deprecated weight arguments.** The original used `densenet161(pretrained=True/False)`. The rebuild uses the `weights` interface - `weights=None` for scratch, `DenseNet161_Weights.IMAGENET1K_V1` for transfer - together with the transforms that ship with those weights.
- **Preprocessing.** The original resized to 224 px and converted to a tensor, omitting the ImageNet normalisation that the pretrained weights expect. The rebuild uses the weights recipe, which inherits the normalisation constants and bilinear interpolation, overridden to a disclosed 64 px resolution so the full-data experiment fits a CPU budget. Both variants share the transform exactly.
- **Frozen backbone.** The original only set `requires_grad = False`, which does not stop `model.train()` from updating the pretrained backbone's BatchNorm running buffers. The rebuild additionally holds `features` in `eval()` after every `train()`, and the optimiser owns only parameters with `requires_grad=True`.
- **Matched budget and selection.** The original trained scratch for 20 epochs and transfer for 5, printing repeated test-set accuracy during training. The rebuild gives both variants the same fixed budget (one epoch by default), selects the checkpoint by development macro F1, and touches the official test split once.
- **Determinism.** Python, NumPy and torch are seeded, the DataLoader shuffle generator is seeded, and both variants share the same data split and the same shuffle seed.

The comparison is still not a clean transfer-learning win: the variants do not have equal trainable parameter counts, as the limitations below and the parameter panel of the comparison figure make explicit.

## Reusable API

`mlshowcase/transfer.py` separates the reusable pieces from the CLI experiment:

- `build_transform` / `describe_transform` - the DenseNet-161 weights recipe at 64 px;
- `build_model` - `scratch` (random init) or `pretrained` (ImageNet weights, frozen backbone, fresh 100-class head);
- `freeze_backbone_for_transfer` / `trainable_parameters` - freezing and the optimiser's parameter list;
- `train_epoch` - one epoch with optional `frozen_prefixes` that keep frozen BatchNorm buffers in evaluation mode;
- `evaluate` - sample-count-weighted loss, accuracy and macro F1 over all 100 classes;
- `run_experiment` - the full protocol used by the CLI.

The cell below prints the preprocessing description recorded in the evidence JSON instead of importing the module, so this read-only notebook renders in the base environment without the deep-learning extras. It builds no model and trains nothing: `build_model("pretrained")` would download the ImageNet checkpoint on a real run, and running the experiment itself requires `uv sync --locked --extra deep-learning`.

In [ ]:
preprocessing = payload["protocol"]["preprocessing"]

print("weights transform shared by both variants, as recorded in the evidence JSON:")
for key in sorted(preprocessing):
    print(f"  {key}: {preprocessing[key]}")

print()
print(
    "The reusable API (build_transform, build_model, freeze_backbone_for_transfer, "
    "trainable_parameters, train_epoch, evaluate, run_experiment) lives in "
    "mlshowcase/transfer.py; importing it requires the optional deep-learning extras "
    "(torch, torchvision)."
)

## Runs, selections and final scores

Each entry in `runs` is one variant and seed: the development history per epoch, the selected epoch, the single evaluation on the official test split, wall-clock timings and the total/trainable/frozen parameter counts. Selection maximises development macro F1 over all 100 classes (an exact tie keeps the earliest epoch), and the test split is scored only after the selected state has been reloaded.

In [ ]:
for variant, records in runs.items():
    for record in records:
        selection = record["selection"]
        final = record["final_test"]
        timings = record["timings_seconds"]
        parameters = record["parameters"]
        print(f"{variant} seed {record['seed']} on {record['device']}")
        print(
            f"  parameters: {parameters['trainable']:,} trainable / "
            f"{parameters['total']:,} total ({parameters['frozen']:,} frozen)"
        )
        print(
            f"  selected  : epoch {selection['epoch']}, "
            f"dev macro F1 {selection['dev_macro_f1']:.4f}"
        )
        print(
            f"  final test: loss {final['loss']:.4f}, accuracy {final['accuracy']:.4f}, "
            f"macro F1 {final['macro_f1']:.4f} on {final['samples']} images"
        )
        print(
            f"  timing    : train {timings['train']:.1f}s, dev {timings['dev']:.1f}s, "
            f"test {timings['test']:.1f}s, total {timings['total']:.1f}s"
        )

In [ ]:
summary = payload["summary"]
print(summary["std_kind"] + ":")
for variant, entry in summary["per_variant"].items():
    final = entry["final_test"]
    print(f"{variant}: seeds {entry['seeds']}")
    for metric in ("accuracy", "macro_f1", "loss"):
        stats = final[metric]
        print(
            f"  test {metric:8s} {stats['mean']:.4f} +/- {stats['std']:.4f} "
            f"(per seed {stats['values']})"
        )
delta = summary["delta_pretrained_minus_scratch"]
print("pretrained minus scratch:", {key: round(value, 4) for key, value in delta.items()})

## Generated figures

`transfer_learning_curves.png` shows development macro F1 and cross-entropy against the epoch for both variants (one line per seed, plus a mean when repeats are recorded). `transfer_comparison.png` shows the final test accuracy and macro F1 on the left and the parameter counts on the right, making the unequal trainable-parameter counts explicit. Both figures are regenerated by the CLI and read from `results/`.

In [ ]:
from IPython.display import Image, display

if "figures" not in payload:
    raise KeyError(
        "the evidence file has no 'figures' entry; regenerate it with "
        "`uv run --locked --extra deep-learning python -m mlshowcase.transfer` from the repository root"
    )

for role, filename in sorted(payload["figures"].items()):
    figure_path = RESULTS_JSON.parent / filename
    if not figure_path.is_file():
        raise FileNotFoundError(
            f"missing figure {figure_path.relative_to(SHOWCASE_ROOT)}; regenerate the evidence "
            "with `uv run --locked --extra deep-learning python -m mlshowcase.transfer` from the repository root"
        )
    print(f"{role}: {figure_path.relative_to(SHOWCASE_ROOT)}")
    display(Image(filename=str(figure_path), width=900))

## Limitations

- **Disclosed budget, not convergence.** The default run is one epoch, as `protocol.budget` records. The test scores are therefore small and must not be read as what either model reaches when fully trained; they are not comparable to published CIFAR-100 or ImageNet benchmarks.
- **64 px, not the standard 224 px.** `crop_size=64` with `resize_size=74` was chosen so that the full 45000/5000/10000 protocol fits a CPU budget. Normalisation and bilinear interpolation are inherited from the weights recipe, but the resolution is a deviation from the checkpoint's published protocol; both variants share it exactly.
- **Unequal trainable parameters.** The pretrained variant optimises only its new head; the scratch variant optimises the whole network. That is what a frozen-backbone recipe deploys, so the comparison measures that recipe rather than an equal-capacity control; the parameter panel in `transfer_comparison.png` shows the gap.
- **No augmentation.** The weights' centre-crop evaluation transform is used for training as well, so neither variant sees random crops or flips.
- **One split across seeds.** Repeated seeds vary weight initialisation and shuffling on the same 45000/5000/10000 split; the split seed is recorded and is not varied.
- **One dataset, one architecture.** The evidence supports statements about this CIFAR-100 / DenseNet-161 protocol only.

## Reproducing the evidence

```sh
uv run --locked --extra deep-learning python -m mlshowcase.transfer --epochs 1 --seeds 42 --data-dir data --device cpu
```

The first run downloads CIFAR-100 and the DenseNet-161 IMAGENET1K_V1 checkpoint; the full 45000/5000/10000 protocol then runs with four torch threads on the device `--device` selects. The default is `auto`: CUDA when the host provides it, CPU otherwise; the reproduction command above pins `--device cpu`, which is how the measured evidence was produced. Selected checkpoints are written to `results/checkpoints/` (git-ignored, not referenced by the JSON evidence). Repeating with `--seeds 42 43 44` reruns both variants per seed and adds repeat statistics to the summary.

- Evidence: [`../results/transfer.json`](../results/transfer.json)
- Reference notes: [`../docs/transfer.md`](../docs/transfer.md)
- Implementation: `mlshowcase/transfer.py`; tests: `tests/test_transfer.py`

This notebook explains and displays the measured evidence; it does not retrain, download, or reproduce historical report scores.